In [ ]:
import pandas as pd
import numpy as np
import pymc as pm
import pytensor.tensor as pt

CRASH_FILE = "/content/bostoncrashsproad20182026.csv"
WEATHER_FILE = "/content/weatherdataboston.csv"

# =====================
# LOAD
# =====================
crash = pd.read_csv(CRASH_FILE)
weather = pd.read_csv(WEATHER_FILE)

# =====================
# PREP CRASH DATA
# =====================
crash["crash_date"] = pd.to_datetime(crash["crash_date"])
crash = crash[crash["crash_date"] >= "2023-05-01"].copy()

daily = (
    crash.groupby(["intersection", "road_surface", "crash_date"])
         .size()
         .rename("crash_count")
         .reset_index()
)

# =====================
# PREP WEATHER
# =====================
weather["date"] = pd.to_datetime(weather["date"])
weather = weather.rename(columns={"date": "crash_date"})

df = daily.merge(weather, on="crash_date", how="left")
df["year"] = df["crash_date"].dt.year

# =====================
# SPLIT
# =====================
train = df[(df["crash_date"] >= "2023-05-01") &
           (df["crash_date"] <= "2025-12-31")].copy()

future = df[df["year"] == 2026].copy()

# =====================
# CLEAN
# =====================
cols = ["tavg","wspd","rhum","prcp"]
train = train.dropna(subset=cols)
future = future.dropna(subset=cols)

# =====================
# STANDARDIZE
# =====================
for col in cols:
    mean = train[col].mean()
    std = train[col].std()
    train[col] = (train[col] - mean) / std
    future[col] = (future[col] - mean) / std

# =====================
# BUILD CATEGORIES (ON TRAIN ONLY)
# =====================
train["intersection"] = train["intersection"].astype("category")
train["road_surface"] = train["road_surface"].astype("category")

future = future[
    future["intersection"].isin(train["intersection"].cat.categories)
]
future = future[
    future["road_surface"].isin(train["road_surface"].cat.categories)
]

future["intersection"] = pd.Categorical(
    future["intersection"],
    categories=train["intersection"].cat.categories
)
future["road_surface"] = pd.Categorical(
    future["road_surface"],
    categories=train["road_surface"].cat.categories
)

train["intersection_id"] = train["intersection"].cat.codes
train["road_surface_id"] = train["road_surface"].cat.codes
future["intersection_id"] = future["intersection"].cat.codes
future["road_surface_id"] = future["road_surface"].cat.codes

# =====================
# TRAIN MODEL
# =====================
coords = {
    "intersection": train["intersection"].cat.categories,
    "road_surface": train["road_surface"].cat.categories
}

with pm.Model(coords=coords) as model:

    # Priors
    alpha = pm.Exponential("alpha", 1)
    intercept = pm.Normal("intercept", 0, 2)
    beta_year = pm.Normal("beta_year", 0, 1)

    beta_tavg = pm.Normal("beta_tavg", 0, 0.5)
    beta_wspd = pm.Normal("beta_wspd", 0, 0.5)
    beta_rhum = pm.Normal("beta_rhum", 0, 0.5)
    beta_prcp = pm.Normal("beta_prcp", 0, 0.5)

    beta_rs = pm.Normal("beta_rs", 0, 1, dims="road_surface")

    sigma_inter = pm.Exponential("sigma_inter", 1)
    inter_offset = pm.Normal("inter_offset", 0, 1, dims="intersection")
    inter_effect = inter_offset * sigma_inter

    mu = pt.exp(
        intercept +
        beta_year * (train["year"].values - 2023) +
        beta_tavg * train["tavg"].values +
        beta_wspd * train["wspd"].values +
        beta_rhum * train["rhum"].values +
        beta_prcp * train["prcp"].values +
        beta_rs[train["road_surface_id"].values] +
        inter_effect[train["intersection_id"].values]
    )

    pm.NegativeBinomial(
        "obs",
        mu=mu,
        alpha=alpha,
        observed=train["crash_count"].values
    )

    trace = pm.sample(1000, tune=1000, chains=4, cores=4, target_accept=0.9)

# =====================
# PREDICT 2026 (SEPARATE GRAPH)
# =====================
with model:

    mu_future = pt.exp(
        intercept +
        beta_year * (future["year"].values - 2023) +
        beta_tavg * future["tavg"].values +
        beta_wspd * future["wspd"].values +
        beta_rhum * future["rhum"].values +
        beta_prcp * future["prcp"].values +
        beta_rs[future["road_surface_id"].values] +
        inter_effect[future["intersection_id"].values]
    )

    post = pm.sample_posterior_predictive(
        trace,
        var_names=[],
        predictions={"crashes": pm.NegativeBinomial.dist(mu_future, alpha)}
    )

pred = post["crashes"]

future["pred_mean"] = pred.mean(axis=0)
future["pred_low"] = np.quantile(pred, 0.05, axis=0)
future["pred_high"] = np.quantile(pred, 0.95, axis=0)

# =====================
# AGGREGATE
# =====================
risk_2026 = (
    future.groupby("intersection")[["pred_mean","pred_low","pred_high"]]
          .mean()
          .reset_index()
          .sort_values("pred_mean", ascending=False)
)

out = "/content/crash_risk_2026_by_intersection.csv"
risk_2026.to_csv(out, index=False)
print("Saved:", out)
print(risk_2026.head(20))


Output()

KeyError: 'crashes'

In [ ]:
from google.colab import files
files.upload()


Saving bostoncrashsproad20182026.csv to bostoncrashsproad20182026.csv


{'bostoncrashsproad20182026.csv': b'crash_date,intersection,road_surface,speed_limit,latitude,longitude\n2018-01-01,BILLINGS STREET,Dry,30.0,42.28257005,-71.03365054\n2018-01-01,AMHERST STREET,Wet,20.0,42.35894243,-71.09350649\n2018-01-01,PARKVIEW RD,Dry,30.0,42.38698649,-71.20381727\n2018-01-01,DORCHESTER AVENUE,Dry,30.0,42.33903706,-71.05788935\n2018-01-01,FLETT RD,Dry,25.0,42.38634626,-71.18111503\n2018-01-01,RAMP-RTS 93 SB/3 SB TO CALLAHAN TUNNEL,Dry,45.0,42.36381849,-71.05821916\n2018-01-01,BROMFIELD STREET,Dry,35.0,42.27771,-71.0125\n2018-01-01,RAMP-RTS 93 SB/3 SB TO RT 90 WB/PURCHASE,Dry,25.0,42.35706265,-71.05143316\n2018-01-01,INTERSTATE 93 Rte 93 S,Wet,45.0,42.3832524,-71.07678897\n2018-01-01,GLEN LANE,Snow,25.0,42.30190903,-71.08819791\n2018-01-01,MCGRATH HIGHWAY Rte SR28,Dry,35.0,42.3932351,-71.08569206\n2018-01-02,WEST ROXBURY PARKWAY,Dry,25.0,42.29674263,-71.14584682\n2018-01-02,POND STREET,Other,45.0,42.1656501,-70.89421513\n2018-01-02,RESERVOIR PARK DRIVE,Ice,,42.158881

In [ ]:
from google.colab import files
files.upload()


Saving weatherdataboston.csv to weatherdataboston.csv


{'weatherdataboston.csv': b'date,tavg,wspd,rhum,newtime,prcp\r\n2018-01-01,-14.7,26.3,48.80736715,,0\r\n2018-01-02,-12.3,20.5,57.34722222,,0\r\n2018-01-03,-6.2,15.1,54.83333333,,0\r\n2018-01-04,-3.6,36.4,77,,34.3\r\n2018-01-05,-7.8,40.0,55.625,,0\r\n2018-01-06,-13.1,35.3,44.20833333,,0\r\n2018-01-07,-15.1,25.9,41.16666667,,0\r\n2018-01-08,-4.3,27.0,46.40277778,,0\r\n2018-01-09,1.8,22.3,64.44444444,,0\r\n2018-01-10,-0.4,11.9,60.13888889,,0\r\n2018-01-11,4.1,18.0,80.36111111,,0\r\n2018-01-12,9.8,23.8,94.5,,15.7\r\n2018-01-13,10.2,32.4,81.48611111,,31.5\r\n2018-01-14,-7.1,18.7,49.31944444,,0\r\n2018-01-15,-8.9,22.0,58.54166667,,0\r\n2018-01-16,-4.2,8.3,61.02777778,,0\r\n2018-01-17,0.2,10.4,86.91666667,,5.3\r\n2018-01-18,-2.1,18.0,74.36111111,,0\r\n2018-01-19,-2.4,15.5,67.02777778,,0\r\n2018-01-20,2.9,23.0,54.47222222,,0\r\n2018-01-21,5.8,9.7,62.91666667,,0\r\n2018-01-22,2.4,7.9,89.65277778,,1.5\r\n2018-01-23,3.5,14.8,96.75543478,,32\r\n2018-01-24,3.6,23.4,70.05555556,,0\r\n2018-01-25,-3.6

In [ ]:
!pip -q uninstall -y numba llvmlite
!pip -q install "numpy==2.1.3" "pymc>=5.10" "pytensor>=2.16" "arviz>=0.17"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 70.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 76.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.4 MB/s eta 0:00:00
